# Comparison: Base GPT-2 vs DAPT Checkpoint

Loads a base GPT-2 model from OpenAI `.pkl` params, loads a DAPT model from a checkpoint created by `save_checkpoint()`, and compares token embedding vectors using cosine similarity; then looks at changes in next-token probabilities for representative texts.


## 1. Get directory paths

In [1]:
import os
import sys
import pickle
from pathlib import Path

import torch
import tiktoken

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").exists() and (p / "notebooks").exists():
            return p
    return start

PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)
print("DATA_DIR:", DATA_DIR)


PROJECT_ROOT: /home/markb/llm-from-scratch
SRC_DIR: /home/markb/llm-from-scratch/src
DATA_DIR: /home/markb/llm-from-scratch/data


## 2. Import modules and establish tokenizer

In [2]:
from llm_from_scratch.models import gpt2
from llm_from_scratch.training import training_utils
from llm_from_scratch.configs import gpt2small_config
from llm_from_scratch.utils import token_analysis as ta

tokenizer = tiktoken.get_encoding("gpt2")
ta.tokenizer = tokenizer


In [3]:
# Device selection: prefer CUDA if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Default file paths (edit if needed)
BASE_PARAMS_PATH = DATA_DIR / "gpt2_openai_params_124M.pkl"
DAPT_CKPT_PATH = DATA_DIR / "TEST_abstracts_epoch_lastsave_step_lastsave.pth"

print("BASE_PARAMS_PATH exists:", BASE_PARAMS_PATH.exists(), BASE_PARAMS_PATH)
print("DAPT_CKPT_PATH exists:", DAPT_CKPT_PATH.exists(), DAPT_CKPT_PATH)

if not BASE_PARAMS_PATH.exists():
    raise FileNotFoundError(f"Base params file not found: {BASE_PARAMS_PATH}")
if not DAPT_CKPT_PATH.exists():
    raise FileNotFoundError(f"DAPT checkpoint file not found: {DAPT_CKPT_PATH}")


Using device: cpu
BASE_PARAMS_PATH exists: True /home/markb/llm-from-scratch/data/gpt2_openai_params_124M.pkl
DAPT_CKPT_PATH exists: True /home/markb/llm-from-scratch/data/TEST_abstracts_epoch_lastsave_step_lastsave.pth


## 3. Load base model from OpenAI .pkl params

In [4]:

base_cfg = dict(gpt2small_config.GPT_CONFIG_124M_OPENAI)
base_model = gpt2.setup_model(base_cfg).to(device)

with open(BASE_PARAMS_PATH, "rb") as f:
    base_params = pickle.load(f)

training_utils.load_weights_into_gpt(base_model, base_params)
base_model.eval()
print("Loaded base model from:", BASE_PARAMS_PATH)
print("Base tok_emb shape:", tuple(base_model.tok_emb.weight.shape))


Loaded base model from: /home/markb/llm-from-scratch/data/gpt2_openai_params_124M.pkl
Base tok_emb shape: (50257, 768)


## 4. Load DAPT model from .pth checkpoint produced by save_checkpoint()
Note DAPT model was made by additional training with biomedical abstracts; see README.md

In [5]:

checkpoint = torch.load(DAPT_CKPT_PATH, map_location=device)

if "model_state_dict" not in checkpoint:
    raise KeyError("Checkpoint does not contain 'model_state_dict'.")

if "run_config" in checkpoint and "model_config" in checkpoint["run_config"]:
    dapt_model_cfg = checkpoint["run_config"]["model_config"]
else:
    # Fallback if run_config is missing
    dapt_model_cfg = base_cfg

dapt_model = gpt2.setup_model(dapt_model_cfg).to(device)
dapt_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
dapt_model.eval()

print("Loaded DAPT model from:", DAPT_CKPT_PATH)
print("DAPT tok_emb shape:", tuple(dapt_model.tok_emb.weight.shape))


Loaded DAPT model from: /home/markb/llm-from-scratch/data/TEST_abstracts_epoch_lastsave_step_lastsave.pth
DAPT tok_emb shape: (50257, 768)


## 5. Model Difference Analysis Sections: Comparing the base (publically available weights) to the DAPT (additional training on biomedical abstracts) model


### 5a. Look at changes at the token embedding layer (+ output layer, as there is weight tying in this model)

In [6]:
# Extract token embedding matrices (base vs DAPT)
embed_initial = base_model.tok_emb.weight.detach().float().cpu().clone()
embed_after = dapt_model.tok_emb.weight.detach().float().cpu().clone()

print("shape_before:", tuple(embed_initial.shape))
print("shape_after:", tuple(embed_after.shape))


shape_before: (50257, 768)
shape_after: (50257, 768)


#### 5a.1 Do pairs of tokens that are closely associated in the biomedical text become closer in embedding in the DAPT model?

In [7]:
# Token-pair cosine similarity checks
import pandas as pd

words_dict = {
    "sp_HER2": tokenizer.encode(" HER2"),
    "HER_sp_2": tokenizer.encode("HER 2"),
    "sp_EGFR": tokenizer.encode(" EGFR"),
    "EG_sp_FR": tokenizer.encode("EG FR"),
    "ERBB2_only_BB2": tokenizer.encode("BB2"),
    "ERBB2_only_spERBB": tokenizer.encode(" ERBB"),
    "sp_kinase": tokenizer.encode(" kinase"),
    "cat vs dog": [tokenizer.encode(" cat")[0], tokenizer.encode(" dog")[0]],
}

resdf = pd.DataFrame(columns=[
    "word", "tokenid1", "token1", "tokenid2", "token2", "cos_sim_openai", "cos_sim_aftertrain", "ratio_afterVSbefore"
])

for word, (tokenid1, tokenid2) in words_dict.items():
    cos_sim_openai = ta.compute_cosine_similarity(tokenid1, tokenid2, embed_initial)
    cos_sim_aftertrain = ta.compute_cosine_similarity(tokenid1, tokenid2, embed_after)
    resdf.loc[len(resdf)] = {
        "word": word,
        "tokenid1": tokenid1,
        "token1": repr(tokenizer.decode([tokenid1])),
        "tokenid2": tokenid2,
        "token2": repr(tokenizer.decode([tokenid2])),
        "cos_sim_openai": cos_sim_openai,
        "cos_sim_aftertrain": cos_sim_aftertrain,
        "ratio_afterVSbefore": cos_sim_aftertrain / cos_sim_openai,
    }

print(resdf.to_string(index=False))


             word  tokenid1 token1  tokenid2 token2  cos_sim_openai  cos_sim_aftertrain  ratio_afterVSbefore
          sp_HER2     24906 ' HER'        17    '2'        0.259792            0.263318             1.013573
         HER_sp_2     16879  'HER'       362   ' 2'        0.171015            0.178956             1.046435
          sp_EGFR     41513  ' EG'     10913   'FR'        0.256762            0.250341             0.974991
         EG_sp_FR      7156   'EG'      8782  ' FR'        0.266662            0.237642             0.891174
   ERBB2_only_BB2     15199   'BB'        17    '2'        0.275225            0.272602             0.990468
ERBB2_only_spERBB     13793  ' ER'     15199   'BB'        0.255247            0.260255             1.019622
        sp_kinase     18967 ' kin'       589  'ase'        0.270267            0.271175             1.003361
       cat vs dog      3797 ' cat'      3290 ' dog'        0.549790            0.535333             0.973704


**RESULT**: Fairly small changes. ' HER' and '2' show a ~ +1% change. 'EG' and 'FR', which would be ' EGFR' show a slight decrease (~ -2%). The control of ' cat' and ' dog' show a small decrease (~ -2%). This implies that there may be larger changes in other parts of the model.

#### 5a.2 Do token embeddings stay basically the same across base model vs DAPT model? 

In [8]:
# Extract token embedding matrices and compute cosine similarity per token for base vs DAPT
base_tok_emb = base_model.tok_emb.weight.detach().float().cpu()
dapt_tok_emb = dapt_model.tok_emb.weight.detach().float().cpu()

if base_tok_emb.shape != dapt_tok_emb.shape:
    raise ValueError(
        f"Embedding shape mismatch: base={tuple(base_tok_emb.shape)} vs dapt={tuple(dapt_tok_emb.shape)}"
    )

cos_scores = ta.cosine_similarity_per_token(base_tok_emb, dapt_tok_emb)
print("cos_scores shape:", tuple(cos_scores.shape))
print("cosine min/max/mean:", cos_scores.min().item(), cos_scores.max().item(), cos_scores.mean().item())


cos_scores shape: (50257,)
cosine min/max/mean: 0.8841298818588257 0.9994819760322571 0.9706948399543762


In [9]:
# 4) Rank top 40 most changed and least changed tokens
TOP_K = 40

most_changed_df = ta.rank_tokens_by_cosine_similarity(
    cosine_scores=cos_scores,
    tokenizer=tokenizer,
    k=TOP_K,
    mode="most_dissimilar",
)

least_changed_df = ta.rank_tokens_by_cosine_similarity(
    cosine_scores=cos_scores,
    tokenizer=tokenizer,
    k=TOP_K,
    mode="most_similar",
)

print("Top 40 MOST changed tokens (lowest cosine):")
print(most_changed_df.to_string(index=False))

print()
print("Top 40 LEAST changed tokens (highest cosine):")
print(least_changed_df.to_string(index=False))


Top 40 MOST changed tokens (lowest cosine):
 tokenid          token  cosine_similarity
     921         ' You'           0.884130
    4705        ' Matt'           0.893998
    3497         ' Get'           0.899295
    5137     ' putting'           0.899304
    6889        ' Make'           0.899364
    1644      ' police'           0.899783
    6035         ' Dan'           0.900174
    2495      ' pretty'           0.900250
    5180       ' Chris'           0.900631
    1639          'You'           0.900823
    4995        ' Mike'           0.901138
    4422        ' Alex'           0.901382
   25508         ' 330'           0.901733
    1526         ' Mar'           0.902345
    5395         ' Jim'           0.902827
    2396           'So'           0.903508
    3932         ' Ben'           0.903555
    6209   ' basically'           0.903645
    7214        ' Take'           0.903873
    3807       ' movie'           0.904015
    1223   ' something'           0.904180
    1532  

**RESULT**: There are some changes and some appear significant, but not in the tokens that seem to be most associated with biomedical work

#### 5a.3 Do next token probabilities shift?

In [10]:

GPROMPTS = [
    "Many families like keeping animals in their homes - we call these pets. Among the most popular pets are dogs and",
    "For breast cancer, trastuzumab, neratinib, and tucatinib are used to treat HER",
    "We visited Greek temples, and saw the names of various Greek gods from Greek mythology, including ZEUS, POSEIDON, APOLLO, HADES and finally one for the brave HER",
    "They told stories of him in the Greek myths, he was famous for his bravery and skill, so his portrait was marked HER",
    "Although HER2 amplification is usually considered in the context of breast cancer, it is clear that it can be amplified in other types of cancer also. In colorectal cancer, a subset of patients are found to have HER2 amplification as indicated by copy number increases in ER",
    "We visited the hospital, which is famous for all the gunshot wounds that come into the emergency room. We were told that our friend was being treated by a doctor in the ER",
    ]

TOP_K_NEXT = 10

base_context_size = int(base_cfg["context_length"])
dapt_context_size = int(dapt_model_cfg["context_length"])

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 300)

for prompt in GPROMPTS:
    base_df = ta.top_next_tokens(base_model, prompt, base_context_size, top_k=TOP_K_NEXT)
    dapt_df = ta.top_next_tokens(dapt_model, prompt, dapt_context_size, top_k=TOP_K_NEXT)

    comparison_df = pd.concat(
        [base_df.add_prefix("base_"), dapt_df.add_prefix("dapt_")],
        axis=1,
    )

    print("PROMPT:", prompt)
    print()
    print("Base vs DAPT top next tokens:")
    print(comparison_df)
    print("\n" + "-" * 80 + "\n")


PROMPT: Many families like keeping animals in their homes - we call these pets. Among the most popular pets are dogs and

Base vs DAPT top next tokens:
   base_rank  base_tokenid  base_token  base_probability  dapt_rank  dapt_tokenid   dapt_token  dapt_probability
0          1         11875     ' cats'          0.940449          1         11875      ' cats'          0.723100
1          2          3797      ' cat'          0.005955          2          6844      ' dogs'          0.082069
2          3         33043  ' rabbits'          0.005161          3         25972  ' chickens'          0.025971
3          4         37793  ' puppies'          0.002652          4          3797       ' cat'          0.015991
4          5           584    ' other'          0.002513          5          8891       ' ham'          0.010124
5          6          8891      ' ham'          0.002231          6           511     ' their'          0.006806
6          7         23214   ' wolves'          0.001777 

**RESULT**: Very interesting result: the control prompt did not change much as to the top most probable tokens, but the prompts aimed at HER2 vs HERO and ERBB vs ER certainly did change. This shows that next token probabilities are clearly shifting, consistent with the loss decrease in the DAPT model in the validation set.